# PyGeoFetch — 09: Complete Geospatial Processing

PyGeoFetch is now a full preprocessing + analysis + post-processing platform.

```
Raw Satellite Data
      ↓
client.preprocess.*   ← atmospheric correction, cloud mask, clip, reproject
      ↓
client.indices.*      ← NDVI, EVI, NDWI, NDBI, TCT, PCA, LST, albedo, texture
      ↓
client.post.*         ← vectorize, smooth, zonal stats, compress, COG
      ↓
client.sar.*          ← despeckle, calibrate, flood map, coherence
      ↓
client.pipeline(...)  ← chain all steps, YAML or Python
      ↓
PyGeoVision AI
```

In [ ]:
from pygeofetch import PyGeoFetch
client = PyGeoFetch(log_level='WARNING')
print('Subsystems:')
for s in ['preprocess','indices','post','sar','batch']:
    methods = [m for m in dir(getattr(client,s)) if not m.startswith('_')]
    print(f'  client.{s:<12} {len(methods)} methods: {methods[:6]}...')

## 1. Preprocessing

In [ ]:
from pathlib import Path
# Assume scene.tif downloaded via PyGeoFetch search+download
# scene = 'output/downloads/aws_earth/S2A_18TWL.../B04.tif'

# A: Atmospheric Correction
# result = client.preprocess.atmos('scene.tif', method='dos1')  # Dark Object Subtraction
# result = client.preprocess.atmos('scene.tif', method='sen2cor')  # Sentinel-2 specific

# B: Cloud Masking
# result = client.preprocess.cloud_mask('scene.tif', method='scl', scl_band='SCL.tif')
# result = client.preprocess.cloud_mask('scene.tif', method='fmask')

# B: Cloud Filling from time series
# result = client.preprocess.cloud_fill('cloudy.tif', time_series=['jan.tif','mar.tif'])

# C: Geometric
# result = client.preprocess.clip('scene.tif', bbox=(-74.1, 40.6, -73.7, 40.9))
# result = client.preprocess.clip('scene.tif', geometry='study_area.geojson')
# result = client.preprocess.reproject('scene.tif', crs='EPSG:4326')

# D: Resampling
# result = client.preprocess.resample('B02_10m.tif', resolution=30)  # 10m → 30m
# result = client.preprocess.pansharpen(pan='pan_15m.tif', ms='ms_60m.tif', method='brovey')

# D: Tiling for AI inference
# result = client.preprocess.tile('scene.tif', tile_size=512, overlap=64)
# tiles = result.metadata['tile_paths']

# H: Compositing
# result = client.preprocess.composite(['jan.tif','feb.tif','mar.tif'], method='median')
# result = client.preprocess.mosaic(['scene1.tif','scene2.tif'], method='first')

print('Preprocessing API — all methods available')
methods = [m for m in dir(client.preprocess) if not m.startswith('_')]
for m in methods: print(f'  client.preprocess.{m}()')

## 2. Spectral Indices (20+)

In [ ]:
indices_ref = {
    'ndvi':     'Vegetation  (NIR-Red)/(NIR+Red)             Veg > 0.3',
    'evi':      'Vegetation  G*(NIR-Red)/(NIR+C1*Red-C2*Blue+L)  Enhanced NDVI',
    'savi':     'Vegetation  (NIR-Red)/(NIR+Red+L)*(1+L)     Soil-adjusted',
    'ndwi':     'Water       (Green-NIR)/(Green+NIR)          Water > 0',
    'mndwi':    'Water       (Green-SWIR1)/(Green+SWIR1)      Better urban separation',
    'ndbi':     'Built-up    (SWIR1-NIR)/(SWIR1+NIR)         Urban > 0',
    'ndsi':     'Snow        (Green-SWIR1)/(Green+SWIR1)      Snow > 0.4',
    'ndmi':     'Moisture    (NIR-SWIR1)/(NIR+SWIR1)         Canopy water content',
    'nbr':      'Fire        (NIR-SWIR2)/(NIR+SWIR2)         Pre-fire baseline',
    'dnbr':     'Burn Sev    NBR_pre - NBR_post               > 0.66 = high severity',
    'tct':      'Transform   Brightness, Greenness, Wetness   3-band output',
    'pca':      'Transform   Principal Components              Dimensionality reduction',
    'texture':  'Texture     GLCM features                    Contrast, homogeneity...',
    'lst':      'Thermal     Land Surface Temperature         Kelvin + Celsius',
    'albedo':   'Reflectance Surface albedo (0-1)             Liang 2001',
    'band_math':'Custom      Arbitrary expression on B[i]     Flexible band math',
    'stack':    'Utility     Stack bands into multi-band TIF  Band combination',
}

print(f"{'Index':<12} {'Category':<12} {'Formula / Notes'}")
print('-'*70)
for idx, desc in indices_ref.items():
    print(f"  {idx:<12} {desc}")

In [ ]:
# Example: NDVI
# result = client.indices.ndvi(red='B04.tif', nir='B08.tif', output='ndvi.tif')
# print(result)

# Example: EVI (better in high-biomass regions)
# result = client.indices.evi(blue='B02.tif', red='B04.tif', nir='B08.tif')

# Example: dNBR burn severity
# result = client.indices.dnbr(
#     pre_nir='pre_B08.tif', pre_swir2='pre_B12.tif',
#     post_nir='post_B08.tif', post_swir2='post_B12.tif',
#     output='dnbr.tif'
# )

# Example: Tasseled Cap Transformation
# result = client.indices.tct(
#     blue='B02.tif', green='B03.tif', red='B04.tif',
#     nir='B08.tif', swir1='B11.tif', swir2='B12.tif',
#     sensor='sentinel2'  # or 'landsat8'
# )
# Produces 3-band raster: [Brightness, Greenness, Wetness]

# Example: PCA
# result = client.indices.pca(
#     inputs=['B02.tif','B03.tif','B04.tif','B08.tif','B11.tif','B12.tif'],
#     n_components=3
# )
# result.metadata['explained_variance_pct']  → [45.2, 28.1, 12.3] %

# Example: Land Surface Temperature
# result = client.indices.lst('B10.tif', emissivity=0.97, sensor='landsat8')
# Band 1 = Kelvin, Band 2 = Celsius

# Example: Custom band arithmetic
# result = client.indices.band_math(
#     inputs=['B04.tif', 'B08.tif'],
#     expression='(B[1] - B[0]) / (B[1] + B[0] + 1e-6)'
# )

print('Indices API — see comments above for usage')

## 3. Post-Processing

In [ ]:
# G: Vector & Raster Post-Processing

# Vectorize classification or index output
# result = client.post.vectorize('ndvi.tif', threshold=0.3, output='vegetation.geojson')

# Smooth / simplify polygons
# result = client.post.smooth('vegetation.geojson', tolerance=0.5)

# Regularize (orthogonalize) building footprints
# result = client.post.regularize('buildings.geojson')

# Zonal statistics
# result = client.post.zonal_stats('ndvi.tif', 'parcels.geojson', output='ndvi_stats.csv')
# Computes: count, mean, median, min, max, std, percentile_25, percentile_75

# Buffer
# result = client.post.buffer('roads.geojson', distance=15)

# Cloud Optimized GeoTIFF
# result = client.post.cog('scene.tif', compress='deflate', blocksize=512)

# Compress
# result = client.post.compress('scene.tif', method='lzw')  # or deflate, zstd

# Add geometry metrics
# result = client.post.add_geometry_metrics('polygons.geojson')
# Adds: area_m2, perimeter_m, compactness

print('Post-processing API — see comments above for usage')

## 4. SAR Processing

In [ ]:
# F: SAR Processing (Sentinel-1, ALOS PALSAR)

# Speckle filtering
# result = client.sar.despeckle('sentinel1_vv.tif', filter='lee', window=5)
# result = client.sar.despeckle('sentinel1_vv.tif', filter='enhanced_lee', window=7)
# result = client.sar.despeckle('sentinel1_vv.tif', filter='frost')
# result = client.sar.despeckle('sentinel1_vv.tif', filter='gamma')

# Radiometric calibration
# result = client.sar.calibrate('sentinel1_dn.tif', output_type='sigma0', in_db=True)
# output_type: sigma0 (normalized), gamma0 (terrain-flat), beta0 (brightness)

# Flood mapping
# result = client.sar.flood_map('sentinel1_post.tif', threshold=-15.0)
# Change-based (pre/post event):
# result = client.sar.flood_map('sentinel1_post.tif', reference='sentinel1_pre.tif')

# Interferometric coherence
# result = client.sar.coherence('slc_20240101.tif', 'slc_20240113.tif', window=7)
# result.metadata['mean_coherence']  → 0.72
# High coherence = stable surface, Low = change (subsidence, vegetation, flood)

print('SAR API — see comments above for usage')

## 5. Pipeline — Chain Steps

In [ ]:
# Python API pipeline builder
# result = (
#     client.pipeline('ndvi-workflow')
#     .atmos(method='dos1')
#     .cloud_mask(method='scl', scl_band='SCL.tif')
#     .clip(bbox=(-74.1, 40.6, -73.7, 40.9))
#     .ndvi(red='B04.tif', nir='B08.tif')
#     .vectorize(threshold=0.3)
#     .smooth(tolerance=0.5)
#     .cog()
#     .run(input='scene.tif', output_dir='./processed/')
# )
# print(result.to_dict())

# YAML pipeline definition
yaml_pipeline = '''
name: full-sentinel2-workflow
description: Atmospheric correction → cloud mask → NDVI → vectorize → COG

steps:
  - atmos:      {method: dos1}
  - cloud_mask: {method: scl, scl_band: SCL.tif}
  - clip:       {bbox: "-74.1,40.6,-73.7,40.9"}
  - reproject:  {crs: EPSG:4326}
  - ndvi:       {red: B04.tif, nir: B08.tif}
  - vectorize:  {threshold: 0.3, format: geojson}
  - smooth:     {tolerance: 0.5}
  - cog:        {compress: deflate}
'''

from pathlib import Path
Path('output/ndvi_workflow.yaml').parent.mkdir(parents=True, exist_ok=True)
Path('output/ndvi_workflow.yaml').write_text(yaml_pipeline)
print('Pipeline YAML written → output/ndvi_workflow.yaml')

# Load and run:
# from pygeofetch.processing.pipeline import ProcessingPipeline
# pl = ProcessingPipeline.from_yaml('output/ndvi_workflow.yaml', engine=client)
# result = pl.run(input='scene.tif', output_dir='./processed/')

# Or via CLI:
# PyGeoFetch proc-pipeline run output/ndvi_workflow.yaml --input scene.tif

# Pipeline templates:
# PyGeoFetch proc-pipeline template ndvi
# PyGeoFetch proc-pipeline template change_detection
# PyGeoFetch proc-pipeline template flood_map
# PyGeoFetch proc-pipeline template urban_mapping
# PyGeoFetch proc-pipeline template sar_analysis
# PyGeoFetch proc-pipeline template land_cover

print('Pipeline API — see comments above for usage')

## 6. Batch Processing

In [ ]:
# Batch process many scenes with the same chain

# results = client.batch_process(
#     inputs=['scene1.tif', 'scene2.tif', 'scene3.tif', 'scene4.tif'],
#     chain=[
#         ('atmos',     {'method': 'dos1'}),
#         ('clip',      {'bbox': (-74.1, 40.6, -73.7, 40.9)}),
#         ('ndvi',      {}),
#         ('cog',       {'compress': 'deflate'}),
#     ],
#     output_dir='./batch_processed/',
#     parallel=4,
# )

# succeeded = [r for r in results if r.success]
# print(f'Batch complete: {len(succeeded)}/{len(results)} succeeded')

# Custom function batch
# def my_pipeline(inp, out_dir, threshold=0.3):
#     result = client.preprocess.clip(inp, bbox=(-74,40,-73,41))
#     ndvi = client.indices.ndvi(result.output_path)
#     return client.post.vectorize(ndvi.output_path, threshold=threshold)

# results = client.batch.apply(my_pipeline, ['s1.tif','s2.tif'], threshold=0.4)

print('Batch API — see comments above for usage')

## 7. Complete CLI Reference

In [ ]:
cli_ref = '''
PREPROCESSING (PyGeoFetch preprocess)
  PyGeoFetch preprocess atmos      scene.tif --method dos1|sen2cor|flaash|6s
  PyGeoFetch preprocess topo-correct scene.tif dem.tif --method cosine|minnaert
  PyGeoFetch preprocess cloud-mask  scene.tif --method scl|fmask|threshold
  PyGeoFetch preprocess cloud-fill  cloudy.tif jan.tif mar.tif
  PyGeoFetch preprocess clip        scene.tif --bbox "minx,miny,maxx,maxy"
  PyGeoFetch preprocess reproject   scene.tif --crs EPSG:4326
  PyGeoFetch preprocess resample    scene.tif --resolution 30 --method bilinear
  PyGeoFetch preprocess pansharpen  pan.tif ms.tif --method brovey|ihs|gram_schmidt
  PyGeoFetch preprocess mosaic      s1.tif s2.tif s3.tif --method first|last|min|max
  PyGeoFetch preprocess composite   *.tif --method median|mean|max|min|best_pixel
  PyGeoFetch preprocess tile        scene.tif --tile-size 512 --overlap 64

SPECTRAL INDICES (PyGeoFetch index)
  PyGeoFetch index ndvi   --red B04.tif --nir B08.tif
  PyGeoFetch index evi    --blue B02.tif --red B04.tif --nir B08.tif
  PyGeoFetch index savi   --red B04.tif --nir B08.tif --soil-l 0.5
  PyGeoFetch index ndwi   --green B03.tif --nir B08.tif
  PyGeoFetch index mndwi  --green B03.tif --swir1 B11.tif
  PyGeoFetch index ndbi   --nir B08.tif --swir1 B11.tif
  PyGeoFetch index ndsi   --green B03.tif --swir1 B11.tif
  PyGeoFetch index ndmi   --nir B08.tif --swir1 B11.tif
  PyGeoFetch index nbr    --nir B08.tif --swir2 B12.tif
  PyGeoFetch index dnbr   --pre-nir B08.tif --pre-swir2 B12.tif \\
                           --post-nir B08.tif --post-swir2 B12.tif
  PyGeoFetch index tct    --blue B02.tif --green B03.tif --red B04.tif \\
                           --nir B08.tif --swir1 B11.tif --swir2 B12.tif
  PyGeoFetch index pca    B02.tif B03.tif B04.tif B08.tif --components 3
  PyGeoFetch index texture B08.tif --window 7 --features contrast,homogeneity
  PyGeoFetch index lst    B10.tif --emissivity 0.97 --sensor landsat8
  PyGeoFetch index albedo B02.tif B03.tif B04.tif B08.tif B11.tif B12.tif
  PyGeoFetch index band-math B04.tif B08.tif --expr '(B[1]-B[0])/(B[1]+B[0]+1e-6)'
  PyGeoFetch index stack  B02.tif B03.tif B04.tif

POST-PROCESSING (PyGeoFetch post)
  PyGeoFetch post vectorize     ndvi.tif --threshold 0.3 --format geojson
  PyGeoFetch post smooth        polygons.geojson --tolerance 0.5
  PyGeoFetch post regularize    buildings.geojson
  PyGeoFetch post zonal-stats   ndvi.tif parcels.geojson --output stats.csv
  PyGeoFetch post buffer        roads.geojson --distance 15
  PyGeoFetch post centroids     polygons.geojson
  PyGeoFetch post geometry-metrics polygons.geojson
  PyGeoFetch post compress      scene.tif --method lzw|deflate|zstd
  PyGeoFetch post cog           scene.tif --compress deflate --blocksize 512

SAR PROCESSING (PyGeoFetch sar)
  PyGeoFetch sar despeckle  sar.tif --filter lee|enhanced_lee|frost|gamma --window 7
  PyGeoFetch sar calibrate  sar.tif --output-type sigma0|gamma0|beta0 --db
  PyGeoFetch sar flood-map  post.tif --threshold -15 [--reference pre.tif]
  PyGeoFetch sar coherence  slc1.tif slc2.tif --window 7

PROCESSING PIPELINES (PyGeoFetch proc-pipeline)
  PyGeoFetch proc-pipeline template ndvi|change_detection|flood_map|urban_mapping
  PyGeoFetch proc-pipeline validate ndvi_workflow.yaml
  PyGeoFetch proc-pipeline run      ndvi_workflow.yaml --input scene.tif
'''
print(cli_ref)

---
## Architecture Summary

| Subsystem | Methods | Category |
|---|---|---|
| `client.preprocess` | atmos, cloud_mask, cloud_fill, clip, reproject, resample, pansharpen, tile, mosaic, composite, topo_correct | A–H |
| `client.indices` | ndvi, evi, savi, ndwi, mndwi, ndbi, ndsi, ndmi, nbr, dnbr, tct, pca, texture, lst, albedo, band_math, stack | E |
| `client.post` | vectorize, smooth, regularize, zonal_stats, buffer, centroids, add_geometry_metrics, compress, cog | G |
| `client.sar` | despeckle, calibrate, flood_map, coherence | F |
| `client.batch` | process, apply | — |
| `client.pipeline()` | chain builder + YAML loader + runner | — |